# Import log files into panda dataframes

In [2]:
import pandas as pd
from glob import glob
import os


# List all .dat files in the current directory
# elephant
dat_file_root = "/srv/data/stratbox_simulations/stratbox_particle_runs/bx5/smd132/sn34/pe300/4pc_resume/4pc"

dat_files = glob(os.path.join(dat_file_root, "SBfeedback.dat"))

# Initialize an empty DataFrame
all_data = pd.DataFrame()

# Read and concatenate data from all .dat files
for dat_file in dat_files:
    # Assuming space-separated values in the .dat files
    df = pd.read_csv(dat_file, delim_whitespace=True, header=None,
                     names=['SBid', 'nSN', 'time',
                            'posx', 'posy', 'posz', 'velx', 'vely', 'velz'])
    
    # Convert the columns to numerical
    df = df.iloc[1:]
    df['SBid'] = df['SBid'].map(int)
    df['nSN'] = df['nSN'].map(int)
    df['time'] = pd.to_numeric(df['time'],errors='coerce')
    df['posx'] = pd.to_numeric(df['posx'],errors='coerce')
    df['posy'] = pd.to_numeric(df['posy'],errors='coerce')
    df['posz'] = pd.to_numeric(df['posz'],errors='coerce')
    df['velx'] = pd.to_numeric(df['velx'],errors='coerce')
    df['vely'] = pd.to_numeric(df['vely'],errors='coerce')
    df['velz'] = pd.to_numeric(df['velz'],errors='coerce')
    
    all_data = pd.concat([all_data, df], ignore_index=True)

all_data.head()


,SBid,nSN,time,posx,posy,posz,velx,vely,velz
0,1,2,1.187223e+14,-3.668975e+20,1.541377e+21,8.263009e+20,0.0,0.0,0.0
1,4,2,1.687630e+14,-5.425110e+20,1.028477e+21,9.180327e+19,0.0,0.0,0.0
2,3,2,1.695509e+14,-3.207244e+20,1.174937e+21,-1.624710e+20,0.0,0.0,0.0
3,2,2,1.718160e+14,-7.567030e+20,-1.320291e+20,2.445658e+20,0.0,0.0,0.0
4,1,3,2.087862e+14,-3.668975e+20,1.541377e+21,8.263009e+20,0.0,0.0,0.0


In [3]:
# convert seconds to Megayears
def seconds_to_megayears(seconds):
    return seconds / (1e6 * 365 * 24 * 3600)

# Convert pixel value to pc
def pixel2pc(coord, x_y_z, top_z = 500):
    if x_y_z == "x":
        return coord - top_z
    elif x_y_z == "y":
        return top_z - coord
    elif x_y_z == "z":
        return coord - top_z
    return coord

def pix_256_2pc(pix_256):
    return pix_256 * (1000 / 256)

def pc2pix_256(pc):
    return pc * (256 / 1000)

def cm2pc(cm):
    return cm * 3.24077929e-19

# filter the DataFrame
def filter_data(df, range_coord):
    return df[(df['posx_pc'] > range_coord[0]) & (df['posx_pc'] < range_coord[0] + range_coord[2]) & 
              (df['posy_pc'] > range_coord[1]) & (df['posy_pc'] < range_coord[1] + range_coord[3]) & 
              (df['posz_pc'] > range_coord[4]) & (df['posz_pc'] < range_coord[5])]

def timestamp2Myr(timestamp):
    return (timestamp - 200) * 0.1 + 191

# Convert time to Megayears
all_data['time_Myr'] = seconds_to_megayears(all_data['time'])

# Convert 'pos' from centimeters to parsecs
all_data['posx_pc'] = cm2pc(all_data['posx'])
all_data['posy_pc'] = cm2pc(all_data['posy'])
all_data['posz_pc'] = cm2pc(all_data['posz'])

# Sort the DataFrame by time in ascending order
all_data.sort_values(by='time_Myr', inplace=True)

In [4]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6700 entries, 0 to 6699
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SBid      6700 non-null   int64  
 1   nSN       6700 non-null   int64  
 2   time      6700 non-null   float64
 3   posx      6700 non-null   float64
 4   posy      6700 non-null   float64
 5   posz      6700 non-null   float64
 6   velx      6700 non-null   float64
 7   vely      6700 non-null   float64
 8   velz      6700 non-null   float64
 9   time_Myr  6700 non-null   float64
 10  posx_pc   6700 non-null   float64
 11  posy_pc   6700 non-null   float64
 12  posz_pc   6700 non-null   float64
dtypes: float64(11), int64(2)
memory usage: 732.8 KB


In [6]:
low_x0, low_y0, low_w, low_h, bottom_z, top_z = -300 , -450, 50, 50, -100, 0
# low_x0, low_y0, low_w, low_h = pixel2pc(low_x0, "x"), pixel2pc(low_y0, "y"), pixel2pc(low_w), pixel2pc(low_h)

In [10]:
start_yr = 191
end_yr = start_yr + 15


SB230_df = all_data[all_data['SBid'] == 230] 
SB230_df
# Filter data based on specified conditions
# filtered_data = all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)]
# filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)],
#                             (low_x0, low_y0, low_w, low_h, bottom_z, top_z))
# # filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)], (-92, -101, 50, 50, -400, 400))


# # Print the resulting DataFrame
# filtered_data
# # filtered_data.iloc[0]["posx_pc"]

,SBid,nSN,time,posx,posy,posz,velx,vely,velz,time_Myr,posx_pc,posy_pc,posz_pc
3966,230,2,6.592355e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,209.042218,85.200055,196.260709,53.400783
3130,230,2,6.592359e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,209.042342,85.200055,196.260709,53.400783
3152,230,3,6.639065e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,210.523361,85.200055,196.260709,53.400783
3986,230,3,6.639066e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,210.523386,85.200055,196.260709,53.400783
4005,230,4,6.685816e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,212.005825,85.200055,196.260709,53.400783
3173,230,4,6.685816e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,212.005825,85.200055,196.260709,53.400783
3199,230,5,6.732567e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,213.488290,85.200055,196.260709,53.400783
4031,230,5,6.732567e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,213.488290,85.200055,196.260709,53.400783
4051,230,6,6.779318e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,214.970754,85.200055,196.260709,53.400783
3219,230,6,6.779318e+15,2.628999e+20,6.055973e+20,1.647776e+20,0.0,0.0,0.0,214.970754,85.200055,196.260709,53.400783


In [ ]:
filtered_data.drop(columns=['posx', 'posy', 'posz', 'time'], inplace=True)
filtered_data.to_csv('SNfeedback_185_200.txt', sep='\t', index=False, encoding='utf-8')

In [18]:
def pc2pix_256(pc):
    return pc * (256 / 1000)

In [20]:
new_posx = pc2pix_256(filtered_data['posx_pc']) + 128
new_posx

6239    42.500000
6254    46.500000
6300    34.500002
6302    43.499998
6345    46.500000
6426    46.500000
6463    34.500002
6516    46.500000
6598    46.500000
6684    46.500000
Name: posx_pc, dtype: float64